# El Niño Teleconnection Precipitation Patterns Analysis
## Investigation of EP vs CP El Niños in CESM piControl and 6ka Simulations

This notebook analyzes the teleconnection patterns of Eastern Pacific (EP) and Central Pacific (CP) El Niños by:
1. Loading SST and precipitation data from CESM simulations
2. Computing SST anomalies
3. Calculating E and C indices using PCA
4. Creating composite precipitation maps based on these indices

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from scipy import stats
import xESMF as xe
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

## 1. Data Loading and Preprocessing

In [ ]:
# Define paths to your CESM data
# Modify these paths according to your data directory structure
sst_pi_path = 'path/to/piControl_SST_data.nc'  # Update with your path
sst_6ka_path = 'path/to/6ka_SST_data.nc'        # Update with your path
precip_pi_path = 'path/to/piControl_PRECIP_data.nc'  # Update with your path
precip_6ka_path = 'path/to/6ka_PRECIP_data.nc'      # Update with your path

# Load SST and precipitation data
print("Loading data...")
sst_pi = xr.open_dataset(sst_pi_path)
sst_6ka = xr.open_dataset(sst_6ka_path)
precip_pi = xr.open_dataset(precip_pi_path)
precip_6ka = xr.open_dataset(precip_6ka_path)

print(f"piControl SST shape: {sst_pi.dims}")
print(f"6ka SST shape: {sst_6ka.dims}")
print(f"piControl Precip shape: {precip_pi.dims}")
print(f"6ka Precip shape: {precip_6ka.dims}")

## 2. Regridding Function

In [ ]:
def regrid(ds, target_ds=None):
    """
    Regrid CESM data to 1x1 degree global grid using bilinear interpolation.
    
    Parameters:
    -----------
    ds : xarray.Dataset
        Input dataset with TLONG and TLAT coordinates
    target_ds : xarray.Dataset, optional
        Target grid definition. If None, uses ds's lat/lon
    
    Returns:
    --------
    xarray.Dataset
        Regridded dataset on 1x1 degree grid
    """
    dr = ds.copy()
    
    # 1. Rename the coordinate names to lon and lat
    # because xESMF has no way to guess variable meaning
    if "TLONG" in ds.coords and "TLAT" in ds.coords:
        dr = dr.rename({"TLONG": "lon", "TLAT": "lat"})
    elif "lon" not in ds.coords or "lat" not in ds.coords:
        print("Warning: Dataset doesn't have expected coordinates")
        print(f"Available coordinates: {list(ds.coords)}")
    
    # 2. Create output grid (1 degree x 1 degree global grid)
    if target_ds is None:
        ds_out = xr.Dataset(
            {
                "lat": (["lat"], np.arange(-90, 90.1, 1.0), {"units": "degrees_north"}),
                "lon": (["lon"], np.arange(-180, 180.0, 1.0), {"units": "degrees_east"}),
            }
        )
    else:
        ds_out = target_ds
    
    # 3. Perform regridding
    print(f"Regridding from {dr.sizes} to {ds_out.sizes}...")
    try:
        regridder = xe.Regridder(dr, ds_out, "bilinear")
        dr_out = regridder(dr)
        print("Regridding complete!")
        return dr_out
    except Exception as e:
        print(f"Error during regridding: {e}")
        return dr

In [ ]:
# Apply regridding to all datasets
print("Regridding SST and Precipitation data...\n")

# Create common target grid
target_grid = xr.Dataset(
    {
        "lat": (["lat"], np.arange(-90, 90.1, 1.0), {"units": "degrees_north"}),
        "lon": (["lon"], np.arange(-180, 180.0, 1.0), {"units": "degrees_east"}),
    }
)

# Regrid all datasets
print("Regridding piControl SST...")
sst_pi_regrid = regrid(sst_pi, target_grid)

print("\nRegridding 6ka SST...")
sst_6ka_regrid = regrid(sst_6ka, target_grid)

print("\nRegridding piControl Precipitation...")
precip_pi_regrid = regrid(precip_pi, target_grid)

print("\nRegridding 6ka Precipitation...")
precip_6ka_regrid = regrid(precip_6ka, target_grid)

print("\nAll regridding complete!")

## 3. Compute SST Anomalies

In [ ]:
def compute_anomalies(ds, time_dim='time'):
    """
    Compute anomalies by removing climatological mean.
    
    Parameters:
    -----------
    ds : xarray.Dataset
        Input dataset
    time_dim : str
        Name of time dimension
    
    Returns:
    --------
    xarray.Dataset
        Dataset with anomalies
    """
    # Calculate climatological mean
    climatology = ds.mean(dim=time_dim)
    
    # Compute anomalies
    anomalies = ds - climatology
    
    return anomalies, climatology

# Compute anomalies for SST
print("Computing SST anomalies...\n")
sst_pi_anom, sst_pi_clim = compute_anomalies(sst_pi_regrid)
sst_6ka_anom, sst_6ka_clim = compute_anomalies(sst_6ka_regrid)

print(f"piControl SST anomalies shape: {sst_pi_anom.dims}")
print(f"6ka SST anomalies shape: {sst_6ka_anom.dims}")

# Display summary statistics
print(f"\npiControl SST anomalies - min: {sst_pi_anom.min().values:.2f}, max: {sst_pi_anom.max().values:.2f}")
print(f"6ka SST anomalies - min: {sst_6ka_anom.min().values:.2f}, max: {sst_6ka_anom.max().values:.2f}")

## 4. Calculate E and C Indices Using PCA

In [ ]:
def calculate_nino_indices(sst_anom, lon_range_e=[150, 360], lon_range_c=[120, 150], 
                          lat_range=[-5, 5], time_dim='time'):
    """
    Calculate EP and CP El Niño indices from SST anomalies using PCA.
    
    Parameters:
    -----------
    sst_anom : xarray.Dataset
        SST anomalies
    lon_range_e : list
        Longitude range for EP Nino region [W, E] in degrees
    lon_range_c : list
        Longitude range for CP Nino region [W, E] in degrees
    lat_range : list
        Latitude range for both regions [S, N] in degrees
    time_dim : str
        Name of time dimension
    
    Returns:
    --------
    dict
        Dictionary containing E and C indices, PCA objects, and regional SST
    """
    
    # Extract SST data variable (handle different possible names)
    sst_var = None
    for var in ['SST', 'TS', 'TEMP', 'temperature']:
        if var in sst_anom.data_vars:
            sst_var = sst_anom[var]
            break
    
    if sst_var is None:
        # Use first data variable
        sst_var = list(sst_anom.data_vars.values())[0]
        print(f"Using variable: {sst_var.name}")
    
    # Select equatorial region and time series
    sst_eq = sst_var.sel(lat=slice(lat_range[0], lat_range[1]))
    
    # Extract EP and CP regions
    sst_ep = sst_eq.sel(lon=slice(lon_range_e[0], lon_range_e[1]))
    sst_cp = sst_eq.sel(lon=slice(lon_range_c[0], lon_range_c[1]))
    
    print(f"EP region shape: {sst_ep.shape}")
    print(f"CP region shape: {sst_cp.shape}")
    
    # Reshape for PCA
    # Convert to 2D: (time, space)
    time_len = sst_ep.sizes[time_dim]
    
    # EP region
    sst_ep_2d = sst_ep.values.reshape(time_len, -1)
    # Remove NaN values
    mask_ep = ~np.isnan(sst_ep_2d).any(axis=0)
    sst_ep_2d = sst_ep_2d[:, mask_ep]
    
    # CP region
    sst_cp_2d = sst_cp.values.reshape(time_len, -1)
    mask_cp = ~np.isnan(sst_cp_2d).any(axis=0)
    sst_cp_2d = sst_cp_2d[:, mask_cp]
    
    print(f"EP region valid points: {sst_ep_2d.shape[1]}")
    print(f"CP region valid points: {sst_cp_2d.shape[1]}")
    
    # Standardize the data
    sst_ep_std = (sst_ep_2d - sst_ep_2d.mean(axis=0)) / (sst_ep_2d.std(axis=0) + 1e-10)
    sst_cp_std = (sst_cp_2d - sst_cp_2d.mean(axis=0)) / (sst_cp_2d.std(axis=0) + 1e-10)
    
    # Apply PCA
    pca_ep = PCA(n_components=3)
    pca_cp = PCA(n_components=3)
    
    pc_ep = pca_ep.fit_transform(sst_ep_std)
    pc_cp = pca_cp.fit_transform(sst_cp_std)
    
    # Get E and C indices from first principal components
    E_index = pc_ep[:, 0]  # EP index
    C_index = pc_cp[:, 0]  # CP index
    
    # Standardize indices
    E_index = (E_index - E_index.mean()) / E_index.std()
    C_index = (C_index - C_index.mean()) / C_index.std()
    
    print(f"\nVariance explained by first PC:")
    print(f"  EP: {pca_ep.explained_variance_ratio_[0]:.2%}")
    print(f"  CP: {pca_cp.explained_variance_ratio_[0]:.2%}")
    
    results = {
        'E_index': E_index,
        'C_index': C_index,
        'pca_ep': pca_ep,
        'pca_cp': pca_cp,
        'sst_ep': sst_ep,
        'sst_cp': sst_cp,
        'pc_ep': pc_ep,
        'pc_cp': pc_cp
    }
    
    return results

print("Calculating E and C indices for piControl...\n")
indices_pi = calculate_nino_indices(sst_pi_anom)

print("\n" + "="*60)
print("Calculating E and C indices for 6ka...\n")
indices_6ka = calculate_nino_indices(sst_6ka_anom)

In [ ]:
# Visualize the indices
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# piControl indices
axes[0, 0].plot(indices_pi['E_index'], label='E index', alpha=0.7)
axes[0, 0].axhline(y=0, color='k', linestyle='--', alpha=0.3)
axes[0, 0].set_title('piControl: E (EP) Index', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('Standardized Index')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].legend()

axes[0, 1].plot(indices_pi['C_index'], label='C index', color='orange', alpha=0.7)
axes[0, 1].axhline(y=0, color='k', linestyle='--', alpha=0.3)
axes[0, 1].set_title('piControl: C (CP) Index', fontsize=12, fontweight='bold')
axes[0, 1].set_ylabel('Standardized Index')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].legend()

# 6ka indices
axes[1, 0].plot(indices_6ka['E_index'], label='E index', alpha=0.7)
axes[1, 0].axhline(y=0, color='k', linestyle='--', alpha=0.3)
axes[1, 0].set_title('6ka: E (EP) Index', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Standardized Index')
axes[1, 0].set_xlabel('Time (months)')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].legend()

axes[1, 1].plot(indices_6ka['C_index'], label='C index', color='orange', alpha=0.7)
axes[1, 1].axhline(y=0, color='k', linestyle='--', alpha=0.3)
axes[1, 1].set_title('6ka: C (CP) Index', fontsize=12, fontweight='bold')
axes[1, 1].set_ylabel('Standardized Index')
axes[1, 1].set_xlabel('Time (months)')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].legend()

plt.tight_layout()
plt.savefig('El_Nino_Indices.png', dpi=300, bbox_inches='tight')
plt.show()

print("Figure saved as 'El_Nino_Indices.png'")

## 5. Create Composite Precipitation Maps

In [ ]:
def create_precipitation_composites(precip_data, E_index, C_index, 
                                   threshold_ep=0.5, threshold_cp=0.5,
                                   time_dim='time'):
    """
    Create composite precipitation maps based on E and C indices.
    
    Parameters:
    -----------
    precip_data : xarray.Dataset
        Precipitation data
    E_index : np.array
        EP El Niño index
    C_index : np.array
        CP El Niño index
    threshold_ep : float
        Threshold for EP events (in standard deviations)
    threshold_cp : float
        Threshold for CP events (in standard deviations)
    time_dim : str
        Name of time dimension
    
    Returns:
    --------
    dict
        Dictionary containing composite maps and event masks
    """
    
    # Extract precipitation variable
    precip_var = None
    for var in ['PRECIP', 'PRECT', 'precip', 'precipitation']:
        if var in precip_data.data_vars:
            precip_var = precip_data[var]
            break
    
    if precip_var is None:
        # Use first data variable
        precip_var = list(precip_data.data_vars.values())[0]
        print(f"Using precipitation variable: {precip_var.name}")
    
    # Create masks for EP and CP events
    ep_positive = E_index > threshold_ep
    ep_negative = E_index < -threshold_ep
    cp_positive = C_index > threshold_cp
    cp_negative = C_index < -threshold_cp
    
    print(f"EP positive events: {ep_positive.sum()}")
    print(f"EP negative events: {ep_negative.sum()}")
    print(f"CP positive events: {cp_positive.sum()}")
    print(f"CP negative events: {cp_negative.sum()}")
    
    # Create composites
    composite_ep_pos = precip_var.isel({time_dim: ep_positive}).mean(dim=time_dim)
    composite_ep_neg = precip_var.isel({time_dim: ep_negative}).mean(dim=time_dim)
    composite_ep_anom = composite_ep_pos - composite_ep_neg
    
    composite_cp_pos = precip_var.isel({time_dim: cp_positive}).mean(dim=time_dim)
    composite_cp_neg = precip_var.isel({time_dim: cp_negative}).mean(dim=time_dim)
    composite_cp_anom = composite_cp_pos - composite_cp_neg
    
    results = {
        'composite_ep_pos': composite_ep_pos,
        'composite_ep_neg': composite_ep_neg,
        'composite_ep_anom': composite_ep_anom,
        'composite_cp_pos': composite_cp_pos,
        'composite_cp_neg': composite_cp_neg,
        'composite_cp_anom': composite_cp_anom,
        'ep_positive_mask': ep_positive,
        'cp_positive_mask': cp_positive
    }
    
    return results

print("Creating precipitation composites for piControl...\n")
composites_pi = create_precipitation_composites(
    precip_pi_regrid, 
    indices_pi['E_index'], 
    indices_pi['C_index']
)

print("\n" + "="*60)
print("Creating precipitation composites for 6ka...\n")
composites_6ka = create_precipitation_composites(
    precip_6ka_regrid, 
    indices_6ka['E_index'], 
    indices_6ka['C_index']
)

## 6. Visualize Composite Precipitation Maps

In [ ]:
def plot_composites(composites, title_prefix, vmin=-5, vmax=5, cmap='RdBu_r'):
    """
    Plot composite precipitation maps.
    
    Parameters:
    -----------
    composites : dict
        Dictionary containing composite maps
    title_prefix : str
        Prefix for plot titles
    vmin, vmax : float
        Colorbar limits
    cmap : str
        Colormap name
    """
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12), 
                              subplot_kw=dict(projection=None))
    
    # EP anomaly
    im1 = axes[0, 0].contourf(composites['composite_ep_anom'].lon, 
                              composites['composite_ep_anom'].lat,
                              composites['composite_ep_anom'].values,
                              levels=20, cmap=cmap, vmin=vmin, vmax=vmax)
    axes[0, 0].set_title(f'{title_prefix}: EP El Niño Precipitation Anomaly', 
                         fontsize=12, fontweight='bold')
    axes[0, 0].set_xlabel('Longitude')
    axes[0, 0].set_ylabel('Latitude')
    plt.colorbar(im1, ax=axes[0, 0], label='Precip Anomaly (mm/day)')
    axes[0, 0].grid(True, alpha=0.3)
    
    # CP anomaly
    im2 = axes[0, 1].contourf(composites['composite_cp_anom'].lon,
                              composites['composite_cp_anom'].lat,
                              composites['composite_cp_anom'].values,
                              levels=20, cmap=cmap, vmin=vmin, vmax=vmax)
    axes[0, 1].set_title(f'{title_prefix}: CP El Niño Precipitation Anomaly',
                         fontsize=12, fontweight='bold')
    axes[0, 1].set_xlabel('Longitude')
    axes[0, 1].set_ylabel('Latitude')
    plt.colorbar(im2, ax=axes[0, 1], label='Precip Anomaly (mm/day)')
    axes[0, 1].grid(True, alpha=0.3)
    
    # EP positive composite
    im3 = axes[1, 0].contourf(composites['composite_ep_pos'].lon,
                              composites['composite_ep_pos'].lat,
                              composites['composite_ep_pos'].values,
                              levels=20, cmap='Blues')
    axes[1, 0].set_title(f'{title_prefix}: EP Positive Phase Precipitation',
                         fontsize=12, fontweight='bold')
    axes[1, 0].set_xlabel('Longitude')
    axes[1, 0].set_ylabel('Latitude')
    plt.colorbar(im3, ax=axes[1, 0], label='Precip (mm/day)')
    axes[1, 0].grid(True, alpha=0.3)
    
    # CP positive composite
    im4 = axes[1, 1].contourf(composites['composite_cp_pos'].lon,
                              composites['composite_cp_pos'].lat,
                              composites['composite_cp_pos'].values,
                              levels=20, cmap='Blues')
    axes[1, 1].set_title(f'{title_prefix}: CP Positive Phase Precipitation',
                         fontsize=12, fontweight='bold')
    axes[1, 1].set_xlabel('Longitude')
    axes[1, 1].set_ylabel('Latitude')
    plt.colorbar(im4, ax=axes[1, 1], label='Precip (mm/day)')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    return fig

# Plot piControl composites
fig_pi = plot_composites(composites_pi, 'piControl')
plt.savefig('Composites_piControl.png', dpi=300, bbox_inches='tight')
plt.show()

# Plot 6ka composites
fig_6ka = plot_composites(composites_6ka, '6ka')
plt.savefig('Composites_6ka.png', dpi=300, bbox_inches='tight')
plt.show()

print("Figures saved!")

## 7. Comparison Between piControl and 6ka

In [ ]:
# Create comparison plots
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# EP Anomaly comparison
im1 = axes[0, 0].contourf(composites_pi['composite_ep_anom'].lon,
                          composites_pi['composite_ep_anom'].lat,
                          composites_pi['composite_ep_anom'].values,
                          levels=20, cmap='RdBu_r', vmin=-5, vmax=5)
axes[0, 0].set_title('piControl: EP El Niño Precip Anomaly', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('Latitude')
plt.colorbar(im1, ax=axes[0, 0], label='mm/day')
axes[0, 0].grid(True, alpha=0.3)

im2 = axes[0, 1].contourf(composites_6ka['composite_ep_anom'].lon,
                          composites_6ka['composite_ep_anom'].lat,
                          composites_6ka['composite_ep_anom'].values,
                          levels=20, cmap='RdBu_r', vmin=-5, vmax=5)
axes[0, 1].set_title('6ka: EP El Niño Precip Anomaly', fontsize=12, fontweight='bold')
plt.colorbar(im2, ax=axes[0, 1], label='mm/day')
axes[0, 1].grid(True, alpha=0.3)

# CP Anomaly comparison
im3 = axes[1, 0].contourf(composites_pi['composite_cp_anom'].lon,
                          composites_pi['composite_cp_anom'].lat,
                          composites_pi['composite_cp_anom'].values,
                          levels=20, cmap='RdBu_r', vmin=-5, vmax=5)
axes[1, 0].set_title('piControl: CP El Niño Precip Anomaly', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Longitude')
axes[1, 0].set_ylabel('Latitude')
plt.colorbar(im3, ax=axes[1, 0], label='mm/day')
axes[1, 0].grid(True, alpha=0.3)

im4 = axes[1, 1].contourf(composites_6ka['composite_cp_anom'].lon,
                          composites_6ka['composite_cp_anom'].lat,
                          composites_6ka['composite_cp_anom'].values,
                          levels=20, cmap='RdBu_r', vmin=-5, vmax=5)
axes[1, 1].set_title('6ka: CP El Niño Precip Anomaly', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Longitude')
plt.colorbar(im4, ax=axes[1, 1], label='mm/day')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('Comparison_EP_CP_Anomalies.png', dpi=300, bbox_inches='tight')
plt.show()

print("Comparison figure saved!")

## 8. Statistical Analysis and Significance

In [ ]:
def compute_statistical_significance(precip_data, E_index, C_index, time_dim='time'):
    """
    Compute t-statistics for composite differences.
    
    Parameters:
    -----------
    precip_data : xarray.Dataset
        Precipitation data
    E_index : np.array
        EP El Niño index
    C_index : np.array
        CP El Niño index
    time_dim : str
        Name of time dimension
    
    Returns:
    --------
    dict
        Dictionary containing t-statistics and p-values
    """
    
    # Extract precipitation variable
    precip_var = None
    for var in ['PRECIP', 'PRECT', 'precip', 'precipitation']:
        if var in precip_data.data_vars:
            precip_var = precip_data[var]
            break
    
    if precip_var is None:
        precip_var = list(precip_data.data_vars.values())[0]
    
    # Create masks
    ep_positive = E_index > 0.5
    ep_negative = E_index < -0.5
    cp_positive = C_index > 0.5
    cp_negative = C_index < -0.5
    
    # Extract positive and negative phases
    ep_pos_data = precip_var.isel({time_dim: ep_positive})
    ep_neg_data = precip_var.isel({time_dim: ep_negative})
    cp_pos_data = precip_var.isel({time_dim: cp_positive})
    cp_neg_data = precip_var.isel({time_dim: cp_negative})
    
    # Compute t-statistics
    ep_mean_pos = ep_pos_data.mean(dim=time_dim)
    ep_mean_neg = ep_neg_data.mean(dim=time_dim)
    ep_std_pos = ep_pos_data.std(dim=time_dim)
    ep_std_neg = ep_neg_data.std(dim=time_dim)
    
    ep_n_pos = ep_pos_data.sizes[time_dim]
    ep_n_neg = ep_neg_data.sizes[time_dim]
    
    # Welch's t-test
    ep_se = np.sqrt(ep_std_pos**2/ep_n_pos + ep_std_neg**2/ep_n_neg)
    ep_t_stat = (ep_mean_pos - ep_mean_neg) / ep_se
    
    # Same for CP
    cp_mean_pos = cp_pos_data.mean(dim=time_dim)
    cp_mean_neg = cp_neg_data.mean(dim=time_dim)
    cp_std_pos = cp_pos_data.std(dim=time_dim)
    cp_std_neg = cp_neg_data.std(dim=time_dim)
    
    cp_n_pos = cp_pos_data.sizes[time_dim]
    cp_n_neg = cp_neg_data.sizes[time_dim]
    
    cp_se = np.sqrt(cp_std_pos**2/cp_n_pos + cp_std_neg**2/cp_n_neg)
    cp_t_stat = (cp_mean_pos - cp_mean_neg) / cp_se
    
    return {
        'ep_t_stat': ep_t_stat,
        'cp_t_stat': cp_t_stat,
        'ep_mean_diff': ep_mean_pos - ep_mean_neg,
        'cp_mean_diff': cp_mean_pos - cp_mean_neg
    }

print("Computing statistical significance...\n")
stats_pi = compute_statistical_significance(precip_pi_regrid, 
                                            indices_pi['E_index'], 
                                            indices_pi['C_index'])
stats_6ka = compute_statistical_significance(precip_6ka_regrid,
                                             indices_6ka['E_index'],
                                             indices_6ka['C_index'])

print("Statistical significance computed!")

In [ ]:
# Plot t-statistics
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# piControl EP t-stat
im1 = axes[0, 0].contourf(stats_pi['ep_t_stat'].lon,
                          stats_pi['ep_t_stat'].lat,
                          stats_pi['ep_t_stat'].values,
                          levels=20, cmap='RdBu_r', vmin=-3, vmax=3)
axes[0, 0].set_title('piControl: EP El Niño t-statistic', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('Latitude')
plt.colorbar(im1, ax=axes[0, 0], label='t-value')
axes[0, 0].grid(True, alpha=0.3)

# 6ka EP t-stat
im2 = axes[0, 1].contourf(stats_6ka['ep_t_stat'].lon,
                          stats_6ka['ep_t_stat'].lat,
                          stats_6ka['ep_t_stat'].values,
                          levels=20, cmap='RdBu_r', vmin=-3, vmax=3)
axes[0, 1].set_title('6ka: EP El Niño t-statistic', fontsize=12, fontweight='bold')
plt.colorbar(im2, ax=axes[0, 1], label='t-value')
axes[0, 1].grid(True, alpha=0.3)

# piControl CP t-stat
im3 = axes[1, 0].contourf(stats_pi['cp_t_stat'].lon,
                          stats_pi['cp_t_stat'].lat,
                          stats_pi['cp_t_stat'].values,
                          levels=20, cmap='RdBu_r', vmin=-3, vmax=3)
axes[1, 0].set_title('piControl: CP El Niño t-statistic', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Longitude')
axes[1, 0].set_ylabel('Latitude')
plt.colorbar(im3, ax=axes[1, 0], label='t-value')
axes[1, 0].grid(True, alpha=0.3)

# 6ka CP t-stat
im4 = axes[1, 1].contourf(stats_6ka['cp_t_stat'].lon,
                          stats_6ka['cp_t_stat'].lat,
                          stats_6ka['cp_t_stat'].values,
                          levels=20, cmap='RdBu_r', vmin=-3, vmax=3)
axes[1, 1].set_title('6ka: CP El Niño t-statistic', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Longitude')
plt.colorbar(im4, ax=axes[1, 1], label='t-value')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('T_Statistics.png', dpi=300, bbox_inches='tight')
plt.show()

print("T-statistics figure saved!")

## 9. Summary Statistics and Analysis

In [ ]:
# Create summary statistics table
summary_data = {
    'Parameter': [
        'E Index Mean',
        'E Index Std',
        'C Index Mean',
        'C Index Std',
        'Correlation E-C',
        'EP Events (>0.5σ)',
        'CP Events (>0.5σ)'
    ],
    'piControl': [
        f"{indices_pi['E_index'].mean():.3f}",
        f"{indices_pi['E_index'].std():.3f}",
        f"{indices_pi['C_index'].mean():.3f}",
        f"{indices_pi['C_index'].std():.3f}",
        f"{np.corrcoef(indices_pi['E_index'], indices_pi['C_index'])[0,1]:.3f}",
        f"{(indices_pi['E_index'] > 0.5).sum()}",
        f"{(indices_pi['C_index'] > 0.5).sum()}"
    ],
    '6ka': [
        f"{indices_6ka['E_index'].mean():.3f}",
        f"{indices_6ka['E_index'].std():.3f}",
        f"{indices_6ka['C_index'].mean():.3f}",
        f"{indices_6ka['C_index'].std():.3f}",
        f"{np.corrcoef(indices_6ka['E_index'], indices_6ka['C_index'])[0,1]:.3f}",
        f"{(indices_6ka['E_index'] > 0.5).sum()}",
        f"{(indices_6ka['C_index'] > 0.5).sum()}"
    ]
}

summary_df = pd.DataFrame(summary_data)
print("\n" + "="*70)
print("SUMMARY STATISTICS")
print("="*70)
print(summary_df.to_string(index=False))
print("="*70)

In [ ]:
# Create scatter plot of E vs C indices
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# piControl
axes[0].scatter(indices_pi['E_index'], indices_pi['C_index'], alpha=0.5, s=20)
axes[0].axhline(y=0, color='k', linestyle='--', alpha=0.3)
axes[0].axvline(x=0, color='k', linestyle='--', alpha=0.3)
axes[0].set_xlabel('E Index (EP El Niño)', fontsize=11)
axes[0].set_ylabel('C Index (CP El Niño)', fontsize=11)
axes[0].set_title('piControl: E vs C Indices', fontsize=12, fontweight='bold')
corr_pi = np.corrcoef(indices_pi['E_index'], indices_pi['C_index'])[0,1]
axes[0].text(0.05, 0.95, f'Correlation: {corr_pi:.3f}', 
             transform=axes[0].transAxes, fontsize=11,
             verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
axes[0].grid(True, alpha=0.3)

# 6ka
axes[1].scatter(indices_6ka['E_index'], indices_6ka['C_index'], alpha=0.5, s=20, color='orange')
axes[1].axhline(y=0, color='k', linestyle='--', alpha=0.3)
axes[1].axvline(x=0, color='k', linestyle='--', alpha=0.3)
axes[1].set_xlabel('E Index (EP El Niño)', fontsize=11)
axes[1].set_ylabel('C Index (CP El Niño)', fontsize=11)
axes[1].set_title('6ka: E vs C Indices', fontsize=12, fontweight='bold')
corr_6ka = np.corrcoef(indices_6ka['E_index'], indices_6ka['C_index'])[0,1]
axes[1].text(0.05, 0.95, f'Correlation: {corr_6ka:.3f}',
             transform=axes[1].transAxes, fontsize=11,
             verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('E_vs_C_Scatter.png', dpi=300, bbox_inches='tight')
plt.show()

print("Scatter plot saved!")

## 10. Key Findings and Interpretation

In [ ]:
print("\n" + "="*70)
print("KEY FINDINGS: EP vs CP El NIÑO TELECONNECTION PATTERNS")
print("="*70)

print("\n1. INDEX CHARACTERISTICS:")
print(f"   piControl E-C correlation: {corr_pi:.3f}")
print(f"   6ka E-C correlation: {corr_6ka:.3f}")
print(f"   Difference: {corr_pi - corr_6ka:.3f}")

print("\n2. EVENT FREQUENCY:")
print(f"   piControl EP events (>0.5σ): {(indices_pi['E_index'] > 0.5).sum()}")
print(f"   6ka EP events (>0.5σ): {(indices_6ka['E_index'] > 0.5).sum()}")
print(f"   piControl CP events (>0.5σ): {(indices_pi['C_index'] > 0.5).sum()}")
print(f"   6ka CP events (>0.5σ): {(indices_6ka['C_index'] > 0.5).sum()}")

print("\n3. PRECIPITATION RESPONSE PATTERNS:")
print("   EP El Niño typically shows:")
print("   - Enhanced precipitation in central Pacific")
print("   - Reduced precipitation in Maritime Continent")
print("   - Strong teleconnections to extratropics")
print("\n   CP El Niño typically shows:")
print("   - Enhanced precipitation near dateline")
print("   - Different teleconnection patterns")
print("   - More symmetric precipitation anomalies")

print("\n4. DIFFERENCES BETWEEN SIMULATIONS:")
print("   - Check if El Niño characteristics differ between piControl and 6ka")
print("   - Evaluate changes in teleconnection strength")
print("   - Assess impacts of orbital forcing on ENSO dynamics")

print("\n" + "="*70)

## 11. Save Results

In [ ]:
# Save indices as NetCDF for future use
print("Saving results...\n")

# Create datasets for indices
indices_pi_ds = xr.Dataset(
    {
        'E_index': (['time'], indices_pi['E_index']),
        'C_index': (['time'], indices_pi['C_index'])
    },
    coords={'time': range(len(indices_pi['E_index']))}
)
indices_pi_ds.to_netcdf('El_Nino_Indices_piControl.nc')
print("Saved: El_Nino_Indices_piControl.nc")

indices_6ka_ds = xr.Dataset(
    {
        'E_index': (['time'], indices_6ka['E_index']),
        'C_index': (['time'], indices_6ka['C_index'])
    },
    coords={'time': range(len(indices_6ka['E_index']))}
)
indices_6ka_ds.to_netcdf('El_Nino_Indices_6ka.nc')
print("Saved: El_Nino_Indices_6ka.nc")

# Save composite anomalies
composites_pi['composite_ep_anom'].to_netcdf('Composite_EP_Anomaly_piControl.nc')
print("Saved: Composite_EP_Anomaly_piControl.nc")

composites_pi['composite_cp_anom'].to_netcdf('Composite_CP_Anomaly_piControl.nc')
print("Saved: Composite_CP_Anomaly_piControl.nc")

composites_6ka['composite_ep_anom'].to_netcdf('Composite_EP_Anomaly_6ka.nc')
print("Saved: Composite_EP_Anomaly_6ka.nc")

composites_6ka['composite_cp_anom'].to_netcdf('Composite_CP_Anomaly_6ka.nc')
print("Saved: Composite_CP_Anomaly_6ka.nc")

print("\nAll results saved successfully!")

In [ ]:
# Create a summary report
report = f"""
================================================================================
                    EL NIÑO TELECONNECTION ANALYSIS REPORT
          Investigation of EP vs CP El Niños in CESM piControl and 6ka
================================================================================

DATA SUMMARY:
- piControl SST: {sst_pi_regrid.dims}
- 6ka SST: {sst_6ka_regrid.dims}
- piControl Precipitation: {precip_pi_regrid.dims}
- 6ka Precipitation: {precip_6ka_regrid.dims}

METHODOLOGY:
1. Regridded SST and precipitation to 1°×1° global grid using bilinear interpolation
2. Computed SST anomalies by removing climatological mean
3. Calculated E (EP) and C (CP) indices using PCA on equatorial regions:
   - EP region: 150°E-180°, 5°S-5°N (Longitude: {indices_pi['sst_ep'].lon.values})
   - CP region: 120°E-150°E, 5°S-5°N (Longitude: {indices_pi['sst_cp'].lon.values})
4. Created composite precipitation maps for positive phases
5. Computed t-statistics for statistical significance

KEY RESULTS:

piControl Simulation:
- E Index statistics: mean={indices_pi['E_index'].mean():.3f}, std={indices_pi['E_index'].std():.3f}
- C Index statistics: mean={indices_pi['C_index'].mean():.3f}, std={indices_pi['C_index'].std():.3f}
- E-C Correlation: {np.corrcoef(indices_pi['E_index'], indices_pi['C_index'])[0,1]:.3f}
- PCA variance (EP): {indices_pi['pca_ep'].explained_variance_ratio_[0]:.2%}
- PCA variance (CP): {indices_pi['pca_cp'].explained_variance_ratio_[0]:.2%}

6ka Simulation:
- E Index statistics: mean={indices_6ka['E_index'].mean():.3f}, std={indices_6ka['E_index'].std():.3f}
- C Index statistics: mean={indices_6ka['C_index'].mean():.3f}, std={indices_6ka['C_index'].std():.3f}
- E-C Correlation: {np.corrcoef(indices_6ka['E_index'], indices_6ka['C_index'])[0,1]:.3f}
- PCA variance (EP): {indices_6ka['pca_ep'].explained_variance_ratio_[0]:.2%}
- PCA variance (CP): {indices_6ka['pca_cp'].explained_variance_ratio_[0]:.2%}

FILES GENERATED:
- El_Nino_Indices.png: Time series of E and C indices
- Composites_piControl.png: Composite precipitation maps for piControl
- Composites_6ka.png: Composite precipitation maps for 6ka
- Comparison_EP_CP_Anomalies.png: Side-by-side comparison of anomalies
- T_Statistics.png: Statistical significance of composite differences
- E_vs_C_Scatter.png: Relationship between E and C indices
- El_Nino_Indices_piControl.nc: E and C indices for piControl
- El_Nino_Indices_6ka.nc: E and C indices for 6ka
- Composite_*_Anomaly_*.nc: Composite precipitation anomaly maps

INTERPRETATION:
The analysis separates EP and CP El Niño events using PCA-based indices and
examines their distinct teleconnection patterns in precipitation. Key differences
between the two modes and between simulations can reveal how orbital forcing
affects ENSO characteristics and their global impacts.

================================================================================
"""

print(report)

# Save report
with open('Analysis_Report.txt', 'w') as f:
    f.write(report)

print("\nReport saved as 'Analysis_Report.txt'")